# 🔍 Exploration — Qualitäts-Muster-Finder

**Ziel dieses Notebooks:**
1. Datensatz kennenlernen
2. `QS.Qualitätsindikator.csv` explorieren → Ziel-Variable bauen
3. Merkmale aus `SO.csv` extrahieren
4. Finale Analysetabelle zusammenführen

**Projektfrage:** Welche Krankenhausmerkmale hängen damit zusammen, dass ein Haus überdurchschnittlich viele Qualitätsprobleme hat?

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# ── Projekt-Root sicherstellen (Notebook liegt in /Notebooks) ────
if Path.cwd().name == 'Notebooks':
    os.chdir(Path.cwd().parent)
print(f'Arbeitsverzeichnis: {Path.cwd()}')

DATA = Path(r"Data/CSV")
print("Verfügbare CSV-Dateien:")
for f in sorted(DATA.glob("*.csv")):
    size_mb = f.stat().st_size / 1_048_576 # Convert bytes to MB
    print(f"  {f.name:45s}  {size_mb:6.1f} MB")

Arbeitsverzeichnis: c:\Users\Admin\VSCode\QualitaetsMusterFinderProjekt
Verfügbare CSV-Dateien:
  AA.csv                                            0.7 MB
  AA.Key.csv                                        0.0 MB
  Abt.Zugang.csv                                    1.2 MB
  Abt301.csv                                        0.0 MB
  Akademische_Lehre.csv                             1.5 MB
  AM.csv                                            3.9 MB
  AM.Key.csv                                        0.0 MB
  AM.Leistung.csv                                   0.0 MB
  AM.VAVU.csv                                       1.9 MB
  AM.VAVU.Key.csv                                   0.0 MB
  AMTS.csv                                          0.3 MB
  AMTS_InstrumentMassnahme.csv                      0.0 MB
  AMTS_Massnahme.csv                                2.4 MB
  AQ.Pflege.csv                                     2.6 MB
  AQ.Ärzte.csv                                      1.3 MB
  AQZF.Key.csv     

## 🔗 Tabellenverbindungen analysieren — Wie hängen die 86 Dateien zusammen?

**Problem:** Der Datensatz besteht aus 86 CSV-Dateien. Es ist unklar, welche Dateien miteinander verknüpft werden können und über welche Spalte (Join-Schlüssel).

**Ansatz:** Wir lesen von jeder Datei nur den Header (keine Daten!) und suchen nach Spaltennamen, die in mehreren Dateien vorkommen. Eine Spalte die in vielen Dateien auftaucht, ist sehr wahrscheinlich ein universeller Join-Schlüssel.

**Warum `nrows=0`?**  
Damit laden wir **nur die Spaltennamen**, ohne eine einzige Datenzeile zu lesen. Das ist auch für die 911 MB große `QS.Qualitätsindikator.csv` in Millisekunden möglich.

In [2]:
from collections import Counter

# Schritt 1: Von jeder CSV nur den Header lesen (nrows=0 = keine Daten!)
alle_spalten = {}
fehler = []

for f in sorted(DATA.glob("*.csv")):
    try:
        df_header = pd.read_csv(f, nrows=0, low_memory=False)
        alle_spalten[f.name] = list(df_header.columns)
    except Exception as e:
        fehler.append(f.name)

print(f"Dateien erfolgreich eingelesen: {len(alle_spalten)}")
print(f"Fehler beim Einlesen:           {len(fehler)}")
if fehler:
    print(f"  → {fehler}")

# Schritt 2: Zählen wie oft jede Spalte in verschiedenen Dateien vorkommt
spalten_zaehler = Counter()
spalte_in_dateien = {}  # welche Dateien haben diese Spalte?

for dateiname, spalten in alle_spalten.items():
    for s in spalten:
        spalten_zaehler[s] += 1
        if s not in spalte_in_dateien:
            spalte_in_dateien[s] = []
        spalte_in_dateien[s].append(dateiname)

# Schritt 3: Alle Spalten, die in mehr als 1 Datei vorkommen → potenzielle Join-Schlüssel
print("\n" + "="*65)
print("SPALTEN DIE IN MEHREREN DATEIEN VORKOMMEN (potenzielle Join-Schlüssel)")
print("="*65)
for spalte, anzahl in spalten_zaehler.most_common():
    if anzahl > 1:
        dateien_liste = ", ".join(spalte_in_dateien[spalte][:5])
        mehr = f" ... (+{anzahl - 5} weitere)" if anzahl > 5 else ""
        print(f"  {spalte:40s}  in {anzahl:3d} Dateien  |  z.B. {dateien_liste}{mehr}")

Dateien erfolgreich eingelesen: 86
Fehler beim Einlesen:           0

SPALTEN DIE IN MEHREREN DATEIEN VORKOMMEN (potenzielle Join-Schlüssel)
  SO.QBID                                   in  34 Dateien  |  z.B. AA.csv, Akademische_Lehre.csv, AMTS.csv, BF.csv, BM.csv ... (+29 weitere)
  ABTID                                     in  11 Dateien  |  z.B. Abt.Zugang.csv, AM.csv, AQ.Pflege.csv, AQ.Ärzte.csv, FA.csv ... (+6 weitere)
  QS.Einrichtung.ID                         in   6 Dateien  |  z.B. QS.Behandlungsumfang.csv, QS.Berufsgruppen.csv, QS.Einrichtungstypen.csv, QS.Pso.csv, QS.Psy.csv ... (+1 weitere)
  SO.Standortnummer                         in   4 Dateien  |  z.B. Konzern.csv, Sicherstellungszuschlaege.csv, Sicherstellungszuschlaege_Fachabteilungen.csv, SO.csv
  AM.VS.Link                                in   3 Dateien  |  z.B. AM.csv, AM.Leistung.csv, AM.VAVU.csv
  MM.KeyPrognose                            in   3 Dateien  |  z.B. Link.csv, MM.csv, MM.Leistungsberechtigung.Prognose

In [3]:
# Schritt 4: Spalten-Präfix-Logik verstehen
#
# ACHTUNG: Hier geht es um den SPALTEN-Präfix (nicht den Dateinamen-Präfix)!
#
# Spaltennamen folgen dem Muster:  PRÄFIX.Beschreibung
#   Beispiele: SO.QBID → Präfix = "SO"
#              FA.Personal.Bereich → Präfix = "FA"
#              QS.Fortbildungspflichtige → Präfix = "QS"
#
# Logik: Wenn Datei A und Datei B beide Spalten mit Präfix "SO" haben,
#        gehören sie thematisch zusammen (beide beschreiben Standort-Daten).
#
# Ergebnis: Wir sehen, welche DATEIEN denselben SPALTEN-PRÄFIX teilen.
# Das gibt uns einen schnellen Überblick über die thematische Gruppierung
# der 86 CSV-Dateien — ohne eine einzige Datenzeile zu lesen.

praefix_zu_dateien = {}
for dateiname, spalten in alle_spalten.items():
    for s in spalten:
        if "." in s:
            spalten_praefix = s.split(".")[0]   # z.B. "SO" aus "SO.QBID"
            if spalten_praefix not in praefix_zu_dateien:
                praefix_zu_dateien[spalten_praefix] = set()
            praefix_zu_dateien[spalten_praefix].add(dateiname)

print("SPALTEN-PRÄFIX-LOGIK — Welche Dateien teilen denselben Spalten-Präfix?")
print("(Spalten-Präfix = der Teil VOR dem Punkt im Spaltennamen, z.B. 'SO' aus 'SO.QBID')")
print("="*70)
for spalten_praefix, dateien in sorted(praefix_zu_dateien.items(), key=lambda x: -len(x[1])):
    if len(dateien) > 1:
        print(f"  Spalten-Präfix '{spalten_praefix}':  {len(dateien):3d} Dateien haben Spalten mit diesem Präfix")
        print(f"    → {', '.join(sorted(dateien)[:6])}")

# Schritt 5: Zusammenfassung — welche Join-Schlüssel werden wir verwenden?
print("\n" + "="*70)
print("ERGEBNIS — VERWENDETE JOIN-SCHLÜSSEL")
print("(Join-Schlüssel = Spaltennamen, die in mehreren Tabellen vorkommen und als Verbindung dienen)")
print("="*70)
join_schluessel = {
    "SO.QBID":  ("universeller Schlüssel — Krankenhaus-ID", spalten_zaehler.get("SO.QBID", 0)),
    "ABTID":    ("Abteilungs-ID (FA.csv ↔ FA.Personalliste.csv)", spalten_zaehler.get("ABTID", 0)),
    "QS.ID":    ("QS-Berichts-ID (QS.csv ↔ QS.Qualitätsindikator.csv)", spalten_zaehler.get("QS.ID", 0)),
}
for schluessel, (beschreibung, n) in join_schluessel.items():
    print(f"  Spalte '{schluessel}'  kommt in {n:2d} Dateien vor  →  {beschreibung}")

SPALTEN-PRÄFIX-LOGIK — Welche Dateien teilen denselben Spalten-Präfix?
(Spalten-Präfix = der Teil VOR dem Punkt im Spaltennamen, z.B. 'SO' aus 'SO.QBID')
  Spalten-Präfix 'SO':   38 Dateien haben Spalten mit diesem Präfix
    → AA.csv, AMTS.csv, Akademische_Lehre.csv, BF.csv, BM.csv, CQ.csv
  Spalten-Präfix 'QS':   10 Dateien haben Spalten mit diesem Präfix
    → QS.Behandlungsumfang.csv, QS.Berufsgruppen.csv, QS.Einrichtungstypen.csv, QS.Fortbildung.csv, QS.Landesrecht.csv, QS.Nachweis.csv
  Spalten-Präfix 'FA':    6 Dateien haben Spalten mit diesem Präfix
    → Abt.Zugang.csv, Abt301.csv, FA.Personalliste.csv, FA.Personen.csv, FA.csv, SO.csv
  Spalten-Präfix 'AM':    6 Dateien haben Spalten mit diesem Präfix
    → AM.Key.csv, AM.Leistung.csv, AM.VAVU.Key.csv, AM.VAVU.csv, AM.csv, LK.Key.csv
  Spalten-Präfix 'Fachexpertise':    4 Dateien haben Spalten mit diesem Präfix
    → AQ.Pflege.csv, AQ.Ärzte.csv, AQZF.Key.csv, PQZP.Key.csv
  Spalten-Präfix 'MM':    4 Dateien haben Spalten mit d

## 1️⃣ Stammdaten — SO.csv (Krankenhäuser)

**Wofür brauchen wir diesen Schritt?**  
`SO.csv` ist die Haupttabelle des gesamten Datensatzes. Sie enthält einen Eintrag pro Krankenhaus-Standort und liefert alle Strukturmerkmale, die wir als **Eingabevariablen (Features X)** für die Analyse brauchen: Wie groß ist das Haus? Wer trägt es? In welchem Bundesland liegt es? Ist es eine Uni-Klinik?

**Was ist der Mehrwert?**  
Ohne `SO.csv` haben wir keine Merkmale. Außerdem enthält sie `SO.QBID` — die universelle ID, über die alle anderen Tabellen verknüpft werden. Das macht `SO.csv` zur **Ankertabelle** des gesamten Datenmodells.

**Auf welche Gedanken soll uns das bringen?**  
- Welche Merkmale könnten mit vielen Qualitätsproblemen zusammenhängen? Große Häuser? Bestimmte Träger? Bestimmte Regionen?
- Gibt es genug Häuser pro Trägerart und Bundesland für eine belastbare Analyse?
- Sind die Merkmale vollständig oder gibt es viele fehlende Werte?

**Bezug zur Aufgabenstellung:**  
Die Aufgabe fragt explizit nach Zusammenhängen zwischen *Strukturmerkmalen* (Betten, Personal, Träger, Region, Uni) und Qualitätsproblemen. `SO.csv` liefert genau diese Strukturmerkmale — sie ist deshalb der logische erste Schritt.

In [4]:
so = pd.read_csv(DATA / "SO.csv", low_memory=False)
print(f"Shape: {so.shape}")
print(f"\nSpalten:")
so.dtypes

Shape: (2310, 49)

Spalten:


Berichtsjahr                           int64
FA.QBID                                int64
IK.Weitere                           float64
SO                                     int64
SO.AkaLehrKH                           int64
SO.Betten                              int64
SO.Dateiname                          object
SO.File                               object
SO.FileKey                            object
SO.FZ.Ambulant                         int64
SO.FZ.StaeB                            int64
SO.FZ.Teil                             int64
SO.FZ.Voll                             int64
SO.Geo.Hausnummer                     object
SO.Geo.Ort                            object
SO.Geo.PLZ                             int64
SO.Geo.Straße                         object
SO.IK                                  int64
SO.IKS                                object
SO.Intern                             object
SO.Kommentar                         float64
SO.Latitude                           object
SO.Longitu

In [46]:
so.head(3)

,Berichtsjahr,FA.QBID,IK.Weitere,SO,SO.AkaLehrKH,SO.Betten,SO.Dateiname,SO.File,SO.FileKey,SO.FZ.Ambulant,...,SO.Gemeinde.Einwohnerzahl,SO.Kreis,SO.Kreis.Einwohnerzahl,SO.Kreis_ohne_Zuordnung,SO.Kreis.Ags,SO.Regierungsbezirk,SO.Regierungsbezirk.Einwohnerzahl,SO.Uniname,KH.Träger,KH.Träger.Art
0,2023,4918,NaN,1,1,157,260100922-773143000-2023,260100922-773143000-2023,260100922-773143000-2023,54,...,25510,Schleswig-Flensburg (Kreis),203799,Schleswig-Flensburg,1059,NaN,NaN,"Universität Schleswig-Holstein, Campus Kiel",HELIOS Fachklinik Schleswig GmbH,privat
1,2023,4934,NaN,1,1,38,260101386-772545000-2023,260101386-772545000-2023,260101386-772545000-2023,2912,...,15288,Ostholstein (Kreis),202014,Ostholstein,1055,NaN,NaN,Medizinische Universität zu Lübeck,Kinderzentrum Pelzerhaken - Sozialpädiatrische...,freigemeinnützig
2,2023,4901,NaN,1,1,214,260100660-772926000-2023,260100660-772926000-2023,260100660-772926000-2023,1892,...,9283,Ostholstein (Kreis),202014,Ostholstein,1055,NaN,NaN,FOM Universität Hamburg,AMEOS Krankenhausgesellschaft Holstein mbH,privat


In [5]:
# Für uns relevante Spalten
merkmale_cols = [
    "SO.QBID", "SO.Name", "SO.Betten",
    "SO.Bundesland", "SO.Uni",
    "KH.Träger", "KH.Träger.Art",
    "SO.Latitude", "SO.Longitude", "SO.Standortnummer"
]
# Nur vorhandene Spalten auswählen
merkmale_cols = [c for c in merkmale_cols if c in so.columns]
so_klein = so[merkmale_cols].copy()
print(f"Krankenhäuser: {so_klein['SO.QBID'].nunique()}")
so_klein.head(5)

Krankenhäuser: 2310


,SO.QBID,SO.Name,SO.Betten,SO.Bundesland,SO.Uni,KH.Träger,KH.Träger.Art,SO.Latitude,SO.Longitude
0,4918,HELIOS Fachklinik Klinik für Erwachsenenpsychi...,157,Schleswig-Holstein,0,HELIOS Fachklinik Schleswig GmbH,privat,"54,523578","9,569341"
1,4934,Kinderzentrum Pelzerhaken gGmbH Sozialpädiatri...,38,Schleswig-Holstein,0,Kinderzentrum Pelzerhaken - Sozialpädiatrische...,freigemeinnützig,"54,088517","10,863304"
2,4901,AMEOS Klinikum Heiligenhafen,214,Schleswig-Holstein,0,AMEOS Krankenhausgesellschaft Holstein mbH,privat,"54,372918","10,963462"
3,4938,Tagesklinik Am Rosenweg Büchen,12,Schleswig-Holstein,0,Diakonie Nord.Nord.Ost in Holstein gemeinnützi...,freigemeinnützig,"53,472654","10,623293"
4,4925,Psychiatrisches Krankenhaus Rickling,360,Schleswig-Holstein,0,Landesverein für Innere Mission in Schleswig-H...,freigemeinnützig,"53,999535","10,174524"


In [6]:
# Überblick Trägerarten
print("Träger.Art:")
print(so_klein["KH.Träger.Art"].value_counts())
print("\nUni-Kliniken:", so_klein["SO.Uni"].value_counts().to_dict())

Träger.Art:
KH.Träger.Art
öffentlich          863
freigemeinnützig    767
privat              650
Name: count, dtype: int64

Uni-Kliniken: {0: 2199, 1: 111}


## 2️⃣ Qualitätsindikatoren — QS.Qualitätsindikator.csv

**Wofür brauchen wir diesen Schritt?**  
Diese Datei ist der Kern der Analyse. Sie enthält für jedes Krankenhaus und jeden Qualitätsindikator eine standardisierte Bewertung: `R*` (rechnerisch auffällig) oder `N*` (nicht auffällig). Aus diesen Bewertungen bauen wir die **Ziel-Variable (y)** — das, was wir erklären wollen.

**Was ist der Mehrwert?**  
Ohne diese Datei hätten wir keine vergleichbare, für alle ~1.900 Häuser einheitlich erhobene Qualitätsaussage. Die Datei ist vom IQTIG im Auftrag des G-BA erstellt — alle Häuser werden nach denselben gesetzlich festgelegten Regeln bewertet. Das ist der einzige Datensatz im Projekt mit dieser Eigenschaft.

**Warum zuerst nur `nrows=5`?**  
Die Datei ist **911 MB** groß — weit über der 50 MB-Grenze von VS Code. Wir lesen zuerst nur 5 Zeilen, um die Spaltenstruktur zu verstehen und die richtige Bewertungsspalte zu finden, bevor wir die volle Datei laden. Das spart Zeit und Arbeitsspeicher.

**Auf welche Gedanken soll uns das bringen?**  
- Welche Spalte enthält die eigentliche Bewertung? (→ `QSErgBewStrukDialog`)
- Was bedeuten die Codes? Gibt es mehr als R\*/N\*?  
- Wie viele Indikatoren hat ein Haus typischerweise?
- Gibt es Indikatoren-Typen, die wir ausschließen müssen?

**Bezug zur Aufgabenstellung:**  
Die Aufgabe fragt: *Welche Häuser haben überdurchschnittlich viele Qualitätsprobleme?* Um das zu messen, brauchen wir eine Bewertungsskala. Diese Datei liefert genau das — sie ist die **einzige Quelle** für diese Information im gesamten Datensatz.

In [7]:
# Datei ist groß (>50MB) — zuerst nur Header & erste Zeilen
qi_pfad = DATA / "QS.Qualitätsindikator.csv"
qi_head = pd.read_csv(qi_pfad, nrows=5, low_memory=False)
print(f"Spaltenanzahl: {len(qi_head.columns)}")
print("\nSpalten:")
for col in qi_head.columns:
    print(f"  {col}")

Spaltenanzahl: 29

Spalten:
  SO.QBID
  QSErgBewStrukDialog
  QSQI.AEKey
  QSQI.ArtDesWertes
  QSQI.Auswertungseinheit
  QSQI.BezugAndereQSErgebnisse
  QSQI.BezugInfektion
  QSQI.BezugZumVerfahren
  QSQI.Bundesdurchschnitt
  QSQI.BundVertrauensbereich
  QSQI.Einheit
  QSQI.EntwVorherigesBerichtsjahr
  QSQI.Ergebnis
  QSQI.ErgebnisMehrfach
  QSQI.FachlicherHinweisIQTIG
  QSQI.FallzahlBeobachteteEreignisse
  QSQI.FallzahlErwarteteEreignisse
  QSQI.FallzahlGrundgesamtheit
  QSQI.Indikator
  QSQI.KHVertrauensbereich
  QSQI.KommentarBeauftragteStelle
  QSQI.KommentarKrankenhaus
  QSQI.Leistungsbereich
  QSQI.Operator
  QSQI.Referenzbereich
  QSQI.Referenzwert
  QSQI.RisikoadjustierteRate
  QSQI.Sortierung
  QSQI.VerglVorherigesBerichtsjahr


In [18]:
qi_head

,SO.QBID,QSErgBewStrukDialog,QSQI.AEKey,QSQI.ArtDesWertes,QSQI.Auswertungseinheit,QSQI.BezugAndereQSErgebnisse,QSQI.BezugInfektion,QSQI.BezugZumVerfahren,QSQI.Bundesdurchschnitt,QSQI.BundVertrauensbereich,...,QSQI.KHVertrauensbereich,QSQI.KommentarBeauftragteStelle,QSQI.KommentarKrankenhaus,QSQI.Leistungsbereich,QSQI.Operator,QSQI.Referenzbereich,QSQI.Referenzwert,QSQI.RisikoadjustierteRate,QSQI.Sortierung,QSQI.VerglVorherigesBerichtsjahr
0,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
1,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
2,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
3,5352,N99,5178,QI,NaN,NaN,False,DeQS,"1,02",1 - 1.04,...,NaN,Zum BJ 2023 erfolgt kein Stellungnahmeverfahren,NaN,PCI Eingriff zur Erweiterung der verengten Her...,NaN,In diesem Berichtsjahr erfolgt für die Qualitä...,NaN,NaN,NaN,eingeschränkt/nicht vergleichbar
4,5352,N99,5178,QI,NaN,NaN,False,DeQS,"1,02",1 - 1.04,...,NaN,Zum BJ 2023 erfolgt kein Stellungnahmeverfahren,NaN,PCI Eingriff zur Erweiterung der verengten Her...,NaN,In diesem Berichtsjahr erfolgt für die Qualitä...,NaN,NaN,NaN,eingeschränkt/nicht vergleichbar


In [8]:
# Vollständige Datei laden
print("Lade QS.Qualitätsindikator.csv ...")
qi = pd.read_csv(qi_pfad, low_memory=False)
print(f"Shape: {qi.shape}")
qi.head(3)

Lade QS.Qualitätsindikator.csv ...
Shape: (417799, 29)


,SO.QBID,QSErgBewStrukDialog,QSQI.AEKey,QSQI.ArtDesWertes,QSQI.Auswertungseinheit,QSQI.BezugAndereQSErgebnisse,QSQI.BezugInfektion,QSQI.BezugZumVerfahren,QSQI.Bundesdurchschnitt,QSQI.BundVertrauensbereich,...,QSQI.KHVertrauensbereich,QSQI.KommentarBeauftragteStelle,QSQI.KommentarKrankenhaus,QSQI.Leistungsbereich,QSQI.Operator,QSQI.Referenzbereich,QSQI.Referenzwert,QSQI.RisikoadjustierteRate,QSQI.Sortierung,QSQI.VerglVorherigesBerichtsjahr
0,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
1,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar
2,5541,R10,5346,QI,NaN,NaN,False,"DeQS, QS-Planung","1,05",0.99 - 1.11,...,0 - 43.21,NaN,NaN,GYN-OP Gynäkologische Operationen (ohne Operat...,<=,"<= 4,18","4,18",NaN,NaN,eingeschränkt/nicht vergleichbar


In [9]:
# Auffällig-Spalte finden
# Suche nach Spalten mit 'auffall', 'auffäll', 'ergebnis', 'bewert'
auffaellig_candidates = [c for c in qi.columns 
                          if any(kw in c.lower() for kw in 
                                 ['auffall', 'auffäll', 'ergebnis', 'bewert', 'ampel', 'status'])]
print("Kandidaten-Spalten für 'auffällig':")
for c in auffaellig_candidates:
    print(f"  {c}: {qi[c].value_counts().head(5).to_dict()}")

Kandidaten-Spalten für 'auffällig':
  QSQI.BezugAndereQSErgebnisse: {52249.0: 11496, 51901.0: 5385, 10211.0: 4422, 51803.0: 3832, 54120.0: 3428}
  QSQI.Ergebnis: {'<=3': 100973, '0': 82564, '100': 15124, '0.84': 584, '0.96': 528}
  QSQI.ErgebnisMehrfach: {False: 417799}


In [10]:
# Alle eindeutigen Werte aller Spalten anzeigen (nur kleine Kardinalität)
for col in qi.columns:
    n_unique = qi[col].nunique()
    if n_unique <= 15:
        print(f"  {col} ({n_unique}): {qi[col].unique().tolist()}")

  QSQI.ArtDesWertes (5): ['QI', 'EKez', 'TKez', 'TKEZ', 'KKez']
  QSQI.Auswertungseinheit (0): [nan]
  QSQI.BezugAndereQSErgebnisse (13): [nan, 52249.0, 50062.0, 51803.0, 231900.0, 10211.0, 54120.0, 2005.0, 2006.0, 2007.0, 50778.0, 50722.0, 181800.0, 51901.0]
  QSQI.BezugInfektion (2): [False, True]
  QSQI.BezugZumVerfahren (3): ['DeQS, QS-Planung', 'DeQS', 'DEQS']
  QSQI.Einheit (2): [nan, 'Punkte', '%']
  QSQI.EntwVorherigesBerichtsjahr (4): ['eingeschränkt/nicht vergleichbar', nan, 'unverändert', 'verbessert', 'verschlechtert']
  QSQI.ErgebnisMehrfach (1): [False]
  QSQI.Operator (2): ['<=', nan, '>=']
  QSQI.Sortierung (12): [nan, 10.0, 11.0, 9.0, 1.0, 3.0, 2.0, 12.0, 8.0, 4.0, 5.0, 6.0, 7.0]
  QSQI.VerglVorherigesBerichtsjahr (4): ['eingeschränkt/nicht vergleichbar', 'unverändert', nan, 'verschlechtert', 'verbessert']


## 3️⃣ Ziel-Variable berechnen

**Wofür brauchen wir diesen Schritt?**  
Jedes Machine-Learning-Modell braucht eine Ziel-Variable — das, was das Modell vorhersagen soll. Hier definieren wir: *Hat ein Krankenhaus überdurchschnittlich viele auffällige Qualitätsindikatoren?* Das Ergebnis ist die binäre Spalte `hat_viele_Probleme` (1 = ja, 0 = nein).

**Was ist der Mehrwert?**  
Wir übersetzen die rohen R\*/N\*-Bewertungen in eine einzige, pro Haus vergleichbare Kennzahl. Die `auffaellig_quote` (Anteil auffälliger Indikatoren) macht Häuser mit sehr unterschiedlichen Indikatorenzahlen vergleichbar.

**Warum diese Designentscheidungen?**
- **Nur `QSQI.ArtDesWertes == 'QI'`** — andere Typen (EKez, TKez, KKez) sind Zählkennzahlen, keine echten Qualitätsindikatoren. Sie würden die Quote verzerren.
- **N99 ausschließen** — `N99` bedeutet „nicht bewertet", z. B. wegen zu wenig Fällen. Das ist **nicht dasselbe** wie „nicht auffällig" — würde man N99 als unauffällig zählen, würde ein Haus mit wenig Daten künstlich gut aussehen.
- **Deduplizierung über `(SO.QBID, QSQI.Indikator)`** — ohne Deduplizierung werden einzelne Indikatoren mehrfach gezählt, was die Quote verfälscht.
- **Median als Schwelle** — der Median ist robuster als der Mittelwert gegenüber Ausreißern und teilt die Häuser in zwei gleich große Gruppen.

**Auf welche Gedanken soll uns das bringen?**  
- Wie hoch ist der Median? (→ 76,92 % — überraschend hoch, d. h. typische Häuser haben ~77 % ihrer Indikatoren im auffälligen Bereich)
- Ist die Verteilung der `auffaellig_quote` symmetrisch oder schief?
- Gibt es Häuser mit 0 % oder 100 % auffälligen Indikatoren?

**Bezug zur Aufgabenstellung:**  
`hat_viele_Probleme` ist **die zentrale Ziel-Variable des gesamten Projekts**. Alle späteren Analysen (Korrelationen, Decision Tree, Visualisierungen) beziehen sich auf genau diese Spalte.

In [11]:
# Erkenntnisse aus Schritt 2:
# - Bewertungsspalte: QSErgBewStrukDialog
#     R* = rechnerisch auffällig  (R10, R20, ...)
#     N99 = nicht bewertet        → ausschließen!
#     N*  = nicht auffällig       (N01, N02, ...)
# - Nur echte QI berücksichtigen: QSQI.ArtDesWertes == 'QI'
#   (EKez, TKez, KKez = Zählkennzahlen, keine Qualitätsindikatoren)
# - QSQI.AEKey ist eine Haus-ID (nicht ein Indikator-Schlüssel!)
#   → Deduplizierung muss über (SO.QBID, QSQI.Indikator) erfolgen

# Schritt 1: nur echte QI-Zeilen
qi_qi = qi[qi["QSQI.ArtDesWertes"] == "QI"].copy()
print(f"Zeilen (nur QI-Typ):            {len(qi_qi):>8,}")

# Schritt 2: nur bewertete Indikatoren (N99 = nicht bewertet → raus)
qi_bewertet = qi_qi[qi_qi["QSErgBewStrukDialog"] != "N99"].copy()
print(f"Zeilen (nach Ausschluss N99):   {len(qi_bewertet):>8,}")

# Schritt 3: Duplikate entfernen — je Haus + Indikator eine Zeile
# QSQI.Indikator = tatsächlicher Indikator-Schlüssel (z.B. "55857")
qi_dedup = qi_bewertet.drop_duplicates(subset=["SO.QBID", "QSQI.Indikator"]).copy()
print(f"Zeilen (nach Deduplizierung):   {len(qi_dedup):>8,}")
print(f"Einzigartige Häuser:            {qi_dedup['SO.QBID'].nunique():>8,}")
print(f"Ø Indikatoren pro Haus:         {len(qi_dedup)/qi_dedup['SO.QBID'].nunique():>8.1f}")

# Schritt 4: auffällig-Flag setzen (R* = auffällig)
qi_dedup["ist_auffaellig"] = qi_dedup["QSErgBewStrukDialog"].str.startswith("R")

# Schritt 5: Quote pro Haus aggregieren
auffaellig_quote = (
    qi_dedup
    .groupby("SO.QBID")
    .agg(
        total_qi       = ("QSQI.Indikator", "count"),
        auffaellig_n   = ("ist_auffaellig", "sum")
    )
    .reset_index()
)
auffaellig_quote["auffaellig_quote"] = auffaellig_quote["auffaellig_n"] / auffaellig_quote["total_qi"]

# Schritt 6: Ziel-Variable — über Median = hat viele Probleme
median_quote = auffaellig_quote["auffaellig_quote"].median()
auffaellig_quote["hat_viele_Probleme"] = (auffaellig_quote["auffaellig_quote"] > median_quote).astype(int)

print(f"\nMedian auffällig-Quote:  {median_quote:.4f}")
print(f"\nZiel-Variable Verteilung:")
print(auffaellig_quote["hat_viele_Probleme"].value_counts())
print(f"\nBeispiel:")
auffaellig_quote.head(8)

Zeilen (nur QI-Typ):             308,726
Zeilen (nach Ausschluss N99):    272,368
Zeilen (nach Deduplizierung):     99,685
Einzigartige Häuser:               1,824
Ø Indikatoren pro Haus:             54.7

Median auffällig-Quote:  0.7692

Ziel-Variable Verteilung:
hat_viele_Probleme
0    925
1    899
Name: count, dtype: int64

Beispiel:


,SO.QBID,total_qi,auffaellig_n,auffaellig_quote,hat_viele_Probleme
0,4876,19,18,0.947368,1
1,4878,2,2,1.000000,1
2,4879,24,17,0.708333,0
3,4880,16,11,0.687500,0
4,4881,30,20,0.666667,0
5,4882,35,25,0.714286,0
6,4886,101,79,0.782178,1
7,4887,99,67,0.676768,0


## 4️⃣ Fortbildungsquote — QS.Fortbildung.csv

**Wofür brauchen wir diesen Schritt?**  
Die Aufgabenstellung nennt *Fortbildungsquote* explizit als potenzielles Strukturmerkmal. Hier berechnen wir es aus den Rohdaten: Wie viel Prozent der fortbildungspflichtigen Ärzte eines Hauses haben ihre Fortbildungspflicht erfüllt?

**Was ist der Mehrwert?**  
Fortbildung ist ein Qualitätsproxy: Häuser, in denen Ärzte ihre Fortbildungspflicht weniger konsequent erfüllen, könnten systematisch schlechtere Qualitätswerte haben. Die Quote macht diesen Aspekt messbar und für die Analyse nutzbar.

**Auf welche Gedanken soll uns das bringen?**  
- Gibt es Häuser mit 0 % oder >100 % Quote? (Datenfehler?)
- Wie ist die Quote zwischen Trägerarten verteilt? Haben private Häuser eine höhere Fortbildungsquote?
- Korreliert eine niedrige Fortbildungsquote mit einer hohen `auffaellig_quote`?

**Bezug zur Aufgabenstellung:**  
Fortbildungsquote ist laut Aufgabenstellung (Fragestellung.docx) ein explizit zu untersuchendes Strukturmerkmal. Dieser Schritt setzt genau das um.

In [12]:
fb = pd.read_csv(DATA / "QS.Fortbildung.csv", low_memory=False)
print(fb.shape)
fb.head(5)

(2310, 4)


,QS.Fortbildungsnachweis_Erbracht_Habende,QS.Fortbildungspflichtige,QS.Nachweispflichtige,SO.QBID
0,41,74,41,4995
1,0,2,0,5014
2,57,75,59,5026
3,1,3,1,5040
4,2,2,2,5049


In [13]:
# Fortbildungsquote berechnen
fb["fortbildungsquote"] = (
    fb["QS.Fortbildungsnachweis_Erbracht_Habende"] / 
    fb["QS.Fortbildungspflichtige"].replace(0, np.nan)
)
fb_quote = fb[["SO.QBID", "fortbildungsquote"]].copy()
print(f"Häuser mit Fortbildungsdaten: {fb_quote['SO.QBID'].nunique()}")
fb_quote.describe()

Häuser mit Fortbildungsdaten: 2310


,SO.QBID,fortbildungsquote
count,2310.000000,2252.000000
mean,6029.500000,0.599528
std,666.983883,0.325935
min,4875.000000,0.000000
25%,5452.250000,0.333333
50%,6029.500000,0.666667
75%,6606.750000,0.875000
max,7184.000000,1.000000


## 5️⃣ Analysetabelle zusammenführen

**Wofür brauchen wir diesen Schritt?**  
Bis hier haben wir Merkmale und Ziel-Variable aus drei verschiedenen Dateien berechnet. Jetzt fügen wir alles über `SO.QBID` zusammen — das Ergebnis ist `analysetabelle.csv`: **eine Zeile pro Krankenhaus, alle Features und die Ziel-Variable in einer Tabelle.**

**Was ist der Mehrwert?**  
Erst durch diesen Join entsteht der eigentliche Analysedatensatz. Ab diesem Punkt können wir Korrelationen berechnen, Visualisierungen erstellen und Modelle trainieren — all das braucht genau diese kompakte Tabellenform.

**Was prüfen wir nach dem Join?**  
- **Zeilenverlust** — wie viele Häuser verlieren wir, weil sie in einer der Quelldateien keine Daten haben?
- **Fehlende Werte** — welche Merkmale haben NaN, und warum? (z. B. Tageskliniken ohne Betten)
- **Plausibilität** — stimmt die Gesamtzahl der Zeilen (~1.900)?

**Auf welche Gedanken soll uns das bringen?**  
- Häuser, die in `QS.Fortbildung.csv` fehlen, haben NaN bei `fortbildungsquote` — sind das systematisch bestimmte Haustypen?
- Der Join-Typ `how='left'` bedeutet: wir behalten alle Häuser aus der Ziel-Variable, auch wenn sie keine Fortbildungsdaten haben.

**Bezug zur Aufgabenstellung:**  
`analysetabelle.csv` ist das **zentrale Ergebnis der Datenvorbereitung**. Alle weiteren Auswertungen, das Dashboard und die Präsentation basieren auf genau dieser Datei.

In [14]:
# Erst ausführen wenn Ziel-Variable aus Schritt 3 vorliegt!

analyse = (
    auffaellig_quote
    .merge(so_klein, on="SO.QBID", how="left")
    .merge(fb_quote,  on="SO.QBID", how="left")
)

print(f"Analysetabelle: {analyse.shape}")
print(f"\nFehlende Werte:\n{analyse.isnull().sum()}")
analyse.head(5)

Analysetabelle: (1824, 14)

Fehlende Werte:
SO.QBID                0
total_qi               0
auffaellig_n           0
auffaellig_quote       0
hat_viele_Probleme     0
SO.Name                0
SO.Betten              0
SO.Bundesland          0
SO.Uni                 0
KH.Träger              0
KH.Träger.Art         28
SO.Latitude            0
SO.Longitude           0
fortbildungsquote     33
dtype: int64


,SO.QBID,total_qi,auffaellig_n,auffaellig_quote,hat_viele_Probleme,SO.Name,SO.Betten,SO.Bundesland,SO.Uni,KH.Träger,KH.Träger.Art,SO.Latitude,SO.Longitude,fortbildungsquote
0,4876,19,18,0.947368,1,Park-Klinik GmbH,39,Schleswig-Holstein,0,Park-Klinik GmbH,privat,"54,326732","10,125615",1.000000
1,4878,2,2,1.000000,1,Johanniter Tagesklinik Schwarzenbek,0,Schleswig-Holstein,0,Johanniter-Krankenhaus Geesthacht GmbH,freigemeinnützig,"53,50572","10,478568",1.000000
2,4879,24,17,0.708333,0,Sankt Elisabeth Krankenhaus Kiel,43,Schleswig-Holstein,0,Lubinus-Kliniken GmbH,freigemeinnützig,"54,316855","10,127381",1.000000
3,4880,16,11,0.687500,0,Malteser Krankenhaus St. Franziskus-Hospital,384,Schleswig-Holstein,0,Malteser Norddeutschland gGmbH,freigemeinnützig,"54,792584","9,420238",0.298507
4,4881,30,20,0.666667,0,Klinikum Nordfriesland gGmbH Inselklinik Föhr-...,18,Schleswig-Holstein,0,Kreis Nordfriesland,öffentlich,"54,685194","8,564174",0.000000


In [23]:
analyse.to_csv("Data/analysetabelle.csv", index=False)
print("Gespeichert: analysetabelle.csv")

Gespeichert: analysetabelle.csv


## 6️⃣ Ärzte pro Bett — FA.Personalliste.csv

**Wofür brauchen wir diesen Schritt?**  
`aerzte_pro_bett` misst die **Personalintensität** eines Hauses: Wie viele Vollzeit-Ärzte kommen auf ein Bett? Häuser mit mehr Ärzten pro Bett haben mehr personelle Kapazität für jeden Patienten — das könnte mit besseren Qualitätswerten zusammenhängen.

**Was ist der Mehrwert?**  
Dieses Merkmal hat sich im Decision Tree als **stärkster Prädiktor** erwiesen (Feature Importance 53,6 % im neu trainierten Modell mit pflege_pro_bett & ist_konzern). Es erklärt mehr Varianz in `hat_viele_Probleme` als alle anderen Merkmale zusammen.

**Warum ist dieser Schritt technisch aufwändiger als die anderen?**  
Weil die Personaldaten nicht direkt mit der Haus-ID verknüpft sind — es braucht **zwei Joins**:
1. `FA.Personalliste.csv` (Personal) → `FA.csv` (Brücke) über `ABTID`
2. `FA.csv` → `SO.csv` über `FA.QBID = SO.QBID`

Außerdem enthält `FA.Personal.Anzahl` Komma-Dezimalzahlen (`"13,47"` statt `13.47`) — das muss vor der Aggregation konvertiert werden, sonst behandelt pandas den Wert als String und die Summe ergibt Unsinn.

**Auf welche Gedanken soll uns das bringen?**  
- Gibt es Häuser mit 0 Betten? (Tageskliniken → `aerzte_pro_bett` = NaN, korrekt)
- Wie ist die Verteilung? Gibt es Ausreißer mit sehr vielen Ärzten pro Bett (Spezialzentren)?
- Korreliert `aerzte_pro_bett` mit Trägerart oder Uni-Status?

**Bezug zur Aufgabenstellung:**  
Personal ist laut Aufgabenstellung ein explizit zu untersuchendes Strukturmerkmal. Dieser Schritt zeigt, dass es das **wichtigste** von allen ist.

In [16]:
# FA.csv: ABTID → FA.QBID (= SO.QBID)
fa = pd.read_csv(DATA / "FA.csv", low_memory=False)
print(f"FA.csv:            {fa.shape}  | Spalten: {list(fa.columns)}")

# FA.Personalliste.csv: Personal pro Abteilung
personal = pd.read_csv(DATA / "FA.Personalliste.csv", low_memory=False)
print(f"FA.Personalliste:  {personal.shape}")
print(f"\nBereich-Werte: {personal['FA.Personal.Bereich'].unique()}")
print(f"Art-Beispiele:  {personal['FA.Personal.Art'].unique()[:8]}")

FA.csv:            (14447, 10)  | Spalten: ['ABTID', 'FA.FZ.Erläuterungen', 'FA.FZ.Teil', 'FA.FZ.Voll', 'FA.Key301', 'FA.Name', 'FA.Ort', 'FA.PLZ', 'FA.QBID', 'FA.Straße']
FA.Personalliste:  (117092, 21)

Bereich-Werte: ['Pflege' 'Ärzte' 'Psych']
Art-Beispiele:  ['Gesundheits- und Krankenpfleger/in' 'Belegärzte'
 'Medizinische/r Fachangestellte/r'
 'Gesundheits- und Kinderkrankenpfleger/in'
 'Operationstechnische/r Assistent/in' 'Altenpfleger/in'
 'Gesundheits- und Krankenpflegehelfer/in' 'Hebammen/Entbindungspfleger']


In [17]:
# Schritt 1: nur Ärzte-Zeilen
aerzte = personal[personal["FA.Personal.Bereich"] == "Ärzte"].copy()

# Schritt 2: Anzahl von Komma-Dezimal → float  (z.B. "13,47" → 13.47)
aerzte["anzahl_float"] = (
    aerzte["FA.Personal.Anzahl"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Schritt 3: summiere Ärzte pro Abteilung → dann pro Haus (FA.QBID)
aerzte_pro_abt = aerzte.groupby("ABTID")["anzahl_float"].sum().reset_index()
aerzte_pro_abt = aerzte_pro_abt.merge(fa[["ABTID", "FA.QBID"]], on="ABTID", how="left")
aerzte_pro_haus = (
    aerzte_pro_abt.groupby("FA.QBID")["anzahl_float"]
    .sum()
    .reset_index()
    .rename(columns={"FA.QBID": "SO.QBID", "anzahl_float": "aerzte_gesamt"})
)

print(f"Häuser mit Ärzte-Daten: {len(aerzte_pro_haus):,}")
print(f"Ø Ärzte pro Haus: {aerzte_pro_haus['aerzte_gesamt'].mean():.1f}")

# Schritt 4: mit SO.Betten mergen → aerzte_pro_bett
aerzte_pro_haus = aerzte_pro_haus.merge(
    so_klein[["SO.QBID", "SO.Betten"]], on="SO.QBID", how="left"
)
aerzte_pro_haus["aerzte_pro_bett"] = (
    aerzte_pro_haus["aerzte_gesamt"] /
    aerzte_pro_haus["SO.Betten"].replace(0, np.nan)
)

print(f"\nBeispiel:")
aerzte_pro_haus.head(6)

Häuser mit Ärzte-Daten: 2,308
Ø Ärzte pro Haus: 112.2

Beispiel:


,SO.QBID,aerzte_gesamt,SO.Betten,aerzte_pro_bett
0,4875,12.00,30,0.400000
1,4876,14.00,39,0.358974
2,4877,6.78,0,NaN
3,4878,6.78,0,NaN
4,4879,34.40,43,0.800000
5,4880,149.65,384,0.389714


In [27]:
# Schritt 5: in Analysetabelle einmergen
analyse = analyse.merge(
    aerzte_pro_haus[["SO.QBID", "aerzte_pro_bett"]],
    on="SO.QBID", how="left"
)

print(f"Analysetabelle jetzt: {analyse.shape}")
print(f"\nFehlende Werte aerzte_pro_bett: {analyse['aerzte_pro_bett'].isna().sum()}")
print(f"  davon SO.Betten == 0: {(analyse['SO.Betten'] == 0).sum()}  (Tageskliniken → NaN korrekt)")
print(f"Ø Ärzte pro Bett: {analyse['aerzte_pro_bett'].mean():.3f}")

# Speichern
analyse.to_csv("Data/analysetabelle.csv", index=False)
print("\n✅ analysetabelle.csv aktualisiert (jetzt 15 Spalten)")

Analysetabelle jetzt: (1824, 16)

Fehlende Werte aerzte_pro_bett: 5
  davon SO.Betten == 0: 4  (Tageskliniken → NaN korrekt)
Ø Ärzte pro Bett: 0.451

✅ analysetabelle.csv aktualisiert (jetzt 15 Spalten)


## 7️⃣ Pflegekräfte pro Bett — SO.Personalliste.csv

**Wofür brauchen wir diesen Schritt?**  
Pflegekräfte pro Bett ist laut `Fragestellung.docx` ein **explizit zu untersuchendes Merkmal** — bisher fehlte es noch in unserer Analysetabelle (offener Punkt in `ToDo.md`).

**Warum SO.Personalliste.csv statt AQ.Pflege.csv?**  
`AQ.Pflege.csv` enthält nur Qualifikationsnachweise (Fachexpertise-Schlüssel), aber **keine Anzahlen**. `SO.Personalliste.csv` hat direkt `SO.QBID` und `SO.Personal.Anzahl` mit dem Bereich `'Pflege'` — damit ist kein Umweg über FA.csv nötig.

**Bezug zur Aufgabenstellung:**  
Ergänzt `aerzte_pro_bett` um die Pflege-Perspektive. Beide zusammen beschreiben die Personalintensität eines Hauses vollständiger.

In [19]:

# SO.Personalliste.csv laden
so_personal = pd.read_csv(DATA / "SO.Personalliste.csv", low_memory=False)
print(f"SO.Personalliste: {so_personal.shape}")
print(f"Bereich-Werte: {so_personal['SO.Personal.Bereich'].unique()}")

# Nur Pflege-Zeilen
pflege = so_personal[so_personal["SO.Personal.Bereich"] == "Pflege"].copy()
print(f"\nPflege-Zeilen: {len(pflege):,}")

# Komma-Dezimal → float (gleiche Logik wie bei Ärzten)
pflege["anzahl_float"] = (
    pflege["SO.Personal.Anzahl"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# Aggregation: Pflege-Vollzeitkräfte pro Haus (SO.QBID direkt verfügbar)
pflege_pro_haus = (
    pflege.groupby("SO.QBID")["anzahl_float"]
    .sum()
    .reset_index()
    .rename(columns={"anzahl_float": "pflege_gesamt"})
)

print(f"\nHäuser mit Pflege-Daten: {len(pflege_pro_haus):,}")
print(f"Ø Pflegekräfte pro Haus: {pflege_pro_haus['pflege_gesamt'].mean():.1f}")

# pflege_pro_bett berechnen
pflege_pro_haus = pflege_pro_haus.merge(
    so_klein[["SO.QBID", "SO.Betten"]], on="SO.QBID", how="left"
)
pflege_pro_haus["pflege_pro_bett"] = (
    pflege_pro_haus["pflege_gesamt"] /
    pflege_pro_haus["SO.Betten"].replace(0, np.nan)
)

print(f"\nØ Pflege pro Bett: {pflege_pro_haus['pflege_pro_bett'].mean():.3f}")
print(f"NaN (Tageskliniken): {pflege_pro_haus['pflege_pro_bett'].isna().sum()}")
pflege_pro_haus.head(5)


SO.Personalliste: (78637, 19)
Bereich-Werte: ['Ärzte' 'Pflege' 'Hygiene' 'Sonstige Ther.' 'Psych']

Pflege-Zeilen: 30,371

Häuser mit Pflege-Daten: 2,310
Ø Pflegekräfte pro Haus: 236.5

Ø Pflege pro Bett: 0.900
NaN (Tageskliniken): 97


,SO.QBID,pflege_gesamt,SO.Betten,pflege_pro_bett
0,4875,16.66,30,0.555333
1,4876,94.19,39,2.415128
2,4877,3.00,0,NaN
3,4878,2.85,0,NaN
4,4879,35.62,43,0.828372


## 8️⃣ Konzernzugehörigkeit — Konzern.csv

**Wofür brauchen wir diesen Schritt?**  
Konzernhäuser (z. B. Helios, Asklepios, AGAPLESION) könnten durch zentrale Qualitätssicherung systematisch andere QI-Profile haben als unabhängige Häuser. Das ist ein potenziell starkes Strukturmerkmal.

**Technischer Hinweis:**  
`Konzern.csv` verwendet `SO.Standortnummer` als Schlüssel, nicht `SO.QBID`. Wir müssen prüfen, ob `SO.Standortnummer` mit einer Spalte in `SO.csv` übereinstimmt.

**Ergebnis:**  
Binäres Merkmal `ist_konzern` (1 = Haus gehört zu einem Konzern, 0 = unabhängig).

In [31]:

# Konzern.csv laden
konzern = pd.read_csv(DATA / "Konzern.csv", low_memory=False)
print(f"Konzern.csv: {konzern.shape}")
print(f"Spalten: {list(konzern.columns)}")
print(f"\nBeispiel:\n{konzern.head(3).to_string()}")

# Schluessel-Check: SO.Standortnummer (Konzern.csv) vs. SO.Standortnummer (SO.csv)
print(f"\nKonzern SO.Standortnummer — Beispielwerte: {konzern['SO.Standortnummer'].head(5).tolist()}")
print(f"SO.csv SO.Standortnummer — Beispielwerte: {so_klein['SO.Standortnummer'].head(5).tolist()}")

# Bugfix 2026-07-29: urspruenglich faelschlich gegen SO.QBID verglichen (0 Treffer).
# SO.csv hat aber eine eigene SO.Standortnummer-Spalte -> damit vergleichen.
treffer = konzern["SO.Standortnummer"].isin(so_klein["SO.Standortnummer"])
print(f"\nTreffer SO.Standortnummer in SO.csv: {treffer.sum()} von {len(konzern)}")

# Binäres Merkmal: ist_konzern
konzern_standortnrn = set(konzern["SO.Standortnummer"].dropna().unique())
konzern_flag = so_klein[["SO.QBID"]].copy()
konzern_flag["ist_konzern"] = so_klein["SO.Standortnummer"].isin(konzern_standortnrn).astype(int)

print(f"\nKonzernhäuser:      {konzern_flag['ist_konzern'].sum():,}")
print(f"Unabhängige Häuser: {(konzern_flag['ist_konzern'] == 0).sum():,}")
konzern_flag.head(5)


Konzern.csv: (1506, 3)
Spalten: ['Konzern', 'Krankenhaus', 'SO.Standortnummer']

Beispiel:
      Konzern                                                                    Krankenhaus  SO.Standortnummer
0  AGAPLESION                             AGAPLESION ALLGEMEINES KRANKENHAUS HAGEN gem. GmbH          772088000
1  AGAPLESION                                    AGAPLESION BETHANIEN KRANKENHAUS HEIDELBERG          771204000
2  AGAPLESION  AGAPLESION BETHESDA KLINIK ULM gGmbH Akademisches Krankenhaus der Universität          773097000

Konzern SO.Standortnummer — Beispielwerte: [772088000, 771204000, 773097000, 772680000, 772234000]
SO.csv SO.Standortnummer — Beispielwerte: [773143000, 772545000, 772926000, 772536000, 773366000]

Treffer SO.Standortnummer in SO.csv: 1395 von 1506

Konzernhäuser:      466
Unabhängige Häuser: 1,844


## 9️⃣ Analysetabelle aktualisieren — alle neuen Merkmale einmergen

Alle drei neuen Merkmale werden jetzt in die bestehende `analysetabelle.csv` eingebunden:
- `pflege_pro_bett` (aus SO.Personalliste.csv)
- `ist_konzern` (aus Konzern.csv, binär 0/1)

In [33]:

# Neue Merkmale in Analysetabelle einmergen
analyse = analyse.merge(
    pflege_pro_haus[["SO.QBID", "pflege_pro_bett"]],
    on="SO.QBID", how="left"
)
analyse = analyse.merge(
    konzern_flag[["SO.QBID", "ist_konzern"]],
    on="SO.QBID", how="left"
)

# ist_konzern NaN (Häuser ohne Match) = 0 (unabhängig)
analyse["ist_konzern"] = analyse["ist_konzern"].fillna(0).astype(int)

print(f"Analysetabelle jetzt: {analyse.shape}  ({analyse.shape[1]} Spalten)")
print(f"\nNeue Spalten:")
print(f"  pflege_pro_bett — fehlend: {analyse['pflege_pro_bett'].isna().sum()}")
print(f"  ist_konzern     — Konzernhäuser: {analyse['ist_konzern'].sum()} ({analyse['ist_konzern'].mean():.1%})")
print(f"\nÜbersicht alle Spalten:")
print(analyse.dtypes.to_string())

# Speichern
analyse.to_csv("Data/analysetabelle.csv", index=False)
print(f"\n✅ analysetabelle.csv gespeichert ({analyse.shape[0]} Zeilen, {analyse.shape[1]} Spalten)")
print("\nNeue vollständige Spalten-Liste:")
for i, col in enumerate(analyse.columns, 1):
    print(f"  {i:2d}. {col}")


Analysetabelle jetzt: (1824, 18)  (18 Spalten)

Neue Spalten:
  pflege_pro_bett — fehlend: 4
  ist_konzern     — Konzernhäuser: 358 (19.6%)

Übersicht alle Spalten:
SO.QBID                 int64
total_qi                int64
auffaellig_n            int64
auffaellig_quote      float64
hat_viele_Probleme      int64
SO.Name                   str
SO.Betten               int64
SO.Bundesland             str
SO.Uni                  int64
KH.Träger                 str
KH.Träger.Art             str
SO.Latitude               str
SO.Longitude              str
SO.Standortnummer       int64
fortbildungsquote     float64
aerzte_pro_bett       float64
pflege_pro_bett       float64
ist_konzern             int64

✅ analysetabelle.csv gespeichert (1824 Zeilen, 18 Spalten)

Neue vollständige Spalten-Liste:
   1. SO.QBID
   2. total_qi
   3. auffaellig_n
   4. auffaellig_quote
   5. hat_viele_Probleme
   6. SO.Name
   7. SO.Betten
   8. SO.Bundesland
   9. SO.Uni
  10. KH.Träger
  11. KH.Träger.Art
  12. 